In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("../")

from ease_recommender import *
from npmi_recommender import *

import pickle as p

def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

d_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
d = cat2idx[d_name]
print(f"{d_name=}")
print(f"Num Rows: {artist_mat[:, d].sum()}")

e_name = find_match_using_terms(["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"], cat2idx)
e = cat2idx[e_name]
print(f"{e_name=}")
print(f"Num Rows: {artist_mat[:, e].sum()}")

loading cache data...
building csr matrices...
done
a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412
c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71
d_name='Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb)'
Num Rows: 12615
e_name='Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)'
Num Rows: 32356


In [7]:
mat = artist_mat
mat = csr_array(mat)

n_users, n_items = mat.shape

X = mat.T @ mat
# X = X / n_users

X.shape

(295860, 295860)

In [259]:
import numpy as np
from scipy import sparse

def sparse_pmi(X, n_users):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    P_x = X.diagonal() / n_users
    P_y = P_x
    
    # Create a copy to store results
    pmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.nonzero()
    
    P_xy = X.data / n_users
    
    denominator = P_x[row_indices] * P_y[col_indices]
    
    pmi_values = np.log2(P_xy / denominator)
    
    pmi.data = pmi_values
    
    # pmi.eliminate_zeros()
    
    return pmi

In [260]:
pmi = sparse_pmi(X, n_users)

In [261]:
import numpy as np
from scipy import sparse

def sparse_pmi_alt(X):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()

    N = X.sum()

    P_xy = X / N
        
    P_x = P_xy.sum(axis=1)
    P_y = P_xy.sum(axis=0)
    
    # Create a copy to store results
    pmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.nonzero()
    
    denominator = P_x[row_indices] * P_y[col_indices]
    
    pmi_values = np.log2(P_xy.data / denominator)
    
    pmi.data = pmi_values
    
    # pmi.eliminate_zeros()
    
    return pmi

In [262]:
pmi_alt = sparse_pmi_alt(X)

In [299]:
# import numpy as np
# from scipy import sparse

# def sparse_pmi_temp_scaling(X, n_users, temp):
#     # Ensure input is CSR for fast row operations
#     if not sparse.isspmatrix_csr(X):
#         X = X.tocsr()
        
#     P_x = X.diagonal() / n_users
#     P_x = P_x**(1/temp)

#     new_sum = P_x.sum()
#     P_x /= new_sum
    
#     P_y = P_x

#     P_xy = X.data / n_users
#     P_xy = P_xy**(1/temp)
#     P_xy /= new_sum

#     assert P_xy.max() <= 1
    
#     # Create a copy to store results
#     pmi = X.copy().astype(np.float32)
    
#     # Get indices of non-zero elements
#     row_indices, col_indices = X.nonzero()
    
#     denominator = P_x[row_indices] * P_y[col_indices]
    
#     pmi_values = np.log2(P_xy / denominator)
    
#     pmi.data = pmi_values
    
#     # pmi.eliminate_zeros()
    
#     return pmi

In [300]:
def local_temp_scaling(p, temp):
    p_logit = logit(p)
    
    scaled_logit = p_logit / temp
    
    return expit(scaled_logit)

In [368]:
# import numpy as np
# from scipy import sparse

# def sparse_pmi_temp_scaling(X, n_users, temp):#, epsilon=1e-9)):
#     # Ensure input is CSR for fast row operations
#     if not sparse.isspmatrix_csr(X):
#         X = X.tocsr()

#     # Get indices of non-zero elements
#     row_indices, col_indices = X.nonzero()

#     # Create a copy to store results
#     pmi = X.copy().astype(np.float32)
    
#     px = X.diagonal() / n_users
#     pxy = X / n_users

#     del X, n_users
    
#     if temp != 1:
#         pxy.data = local_temp_scaling(pxy.data, temp)
#         px = local_temp_scaling(px, temp)
    
#     denominator = px[row_indices] * px[col_indices]
    
#     pmi_values = np.log2(pxy.data / denominator)
#     # pmi_values[pmi_values < 0] = 0
    
#     pmi.data = pmi_values
#     # pmi.data = pmi_values/-np.log2(pxy.data)
    
#     # pmi.data[~np.isfinite(pmi.data)] = -np.inf
    
#     # pmi.eliminate_zeros()
    
#     return pmi

In [383]:
import numpy as np
from scipy import sparse

def sparse_pmi_temp_scaling(pxy, row_indices, col_indices, temp):
    pxy = pxy.copy()
    pxy.data = pxy.data**(1/temp)
    pxy /= pxy.sum()

    px = pxy.sum(axis=1)
    
    denominator = px[row_indices] * px[col_indices]
    
    pmi = pxy.copy()
    pmi_values = np.log2(pxy.data / denominator)
    pmi.data = pmi_values
    
    return pmi

In [371]:
def get_metric(pmi, i, j):
    scores = pmi[i].toarray()

    return np.argsort(-scores).tolist().index(j)

In [380]:
row_indices, col_indices = X.tocoo().nonzero()

In [381]:
pxy = X / n_users

In [390]:
pmi_scaled = sparse_pmi_temp_scaling(pxy, row_indices, col_indices, 1/20)

metrics = [
    get_metric(pmi_scaled, a, b),
    get_metric(pmi_scaled, b, a),
    
    get_metric(pmi_scaled, b, c),
    get_metric(pmi_scaled, c, b),
]

np.mean(metrics)

np.float64(1864.0)

In [366]:
pmi_scaled = sparse_pmi_temp_scaling(X, n_users, 1/10)

In [367]:
metrics = [
    get_metric(pmi_scaled, a, b),
    get_metric(pmi_scaled, b, a),
    
    get_metric(pmi_scaled, b, c),
    get_metric(pmi_scaled, c, b),
]

np.mean(metrics)

np.float64(1264.25)

In [251]:
metrics = [
    get_metric(pmi, a, b),
    get_metric(pmi, b, a),
    
    get_metric(pmi, b, c),
    get_metric(pmi, c, b),
]

np.mean(metrics)

np.float64(1259.75)

In [252]:
metrics = [
    get_metric(pmi_alt, a, b),
    get_metric(pmi_alt, b, a),
    
    get_metric(pmi_alt, b, c),
    get_metric(pmi_alt, c, b),
]

np.mean(metrics)

np.float64(1225.75)

In [237]:
px = X.diagonal()
px = px / px.sum()

pxy = X / n_users

pmi = pxy / (px[:, None] * px[None, :])

MemoryError: Unable to allocate 652. GiB for an array with shape (295860, 295860) and data type float64

In [ ]:
px = X.diagonal()
px = px / px.sum()

pxy = X / n_users

pmi = pxy / (px[:, None] * px[None, :])

In [ ]:
pmi.shape

In [8]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [15]:
alpha = .1 * .85

sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True)

metric = (np.argsort(-sppmi[:, a].toarray()).tolist().index(b) + np.argsort(-sppmi[:, b].toarray()).tolist().index(a))/2
metric

TypeError: sparse_laplace_sppmi() missing 1 required positional argument: 'n_users'

In [21]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, n_users, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    """
    Calculates SPPMI using n_users as the normalization constant (N).
    
    Args:
        X (scipy.sparse.csr_matrix): The input matrix (e.g., User-Item or Co-occurrence).
        n_users (int): The number of users (or documents/transactions) to be used as 
                       the base count 'N' for probability calculations.
        alpha (float): Laplace smoothing additive constant.
        normalize (bool): Whether to normalize PMI by -log(P(x,y)).
        zero_diag (bool): Whether to zero out the diagonal.
        non_neg (bool): Whether to keep only positive PMI values (SPPMI).
    """
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    rows, cols = X.shape
    
    # 2. Calculate "Smoothed" Marginals with n_users as the base
    # N_raw is now n_users (instead of X.sum())
    # We still pretend we added alpha to every cell in the matrix.
    
    # Virtual Total N = Real Users + (alpha * Total Possible Cells)
    # This adapts the additive logic: the denominator now balances 
    # the user count against the smoothed mass.
    N_smoothed = n_users + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    # Note: This assumes X is a contingency table where row_sum is the valid marginal count.
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # The logic remains: sum(row) + sum(alphas_in_row)
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) using the new N_smoothed
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    # 5. Optional Normalization: NPMI
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 6. Update the matrix data
    sppmi.data = pmi_values
    
    # 7. Shifted / Non-negative logic
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up
    sppmi.eliminate_zeros()
    
    return sppmi

In [22]:
alpha = .1 * .85

sppmi = sparse_laplace_sppmi(X, n_users, alpha=alpha, non_neg=True)

metric = (np.argsort(-sppmi[:, a].toarray()).tolist().index(b) + np.argsort(-sppmi[:, b].toarray()).tolist().index(a))/2
metric

10.0

In [23]:
PXY = X / n_users

In [149]:
# def get_scores(i, pop_weight):
#     scores = np.log2(PXY[:, i].toarray())
#     scores[~np.isfinite(scores)] = -np.inf
#     scores = scores*pop_weight + sppmi[:, i].toarray()

#     return scores

In [188]:
# def get_scores(i, pop_weight):
#     scores = np.log2(PXY[:, i].toarray())
#     scores[~np.isfinite(scores)] = -np.inf
#     scores = scores*pop_weight + np.log2(sppmi[:, i].toarray())

#     return scores

In [195]:
# def get_scores(i, pop_weight):
#     scores = PXY[:, i].toarray()
#     scores = scores**pop_weight * sppmi[:, i].toarray()

#     return scores

In [230]:
def get_scores(i, pop_weight):
    scores = PXY[:, i].toarray()
    scores = scores**pop_weight * np.log2(sppmi[:, i].toarray())

    return scores

In [231]:
def get_metric(i, j, pop_weight):
    scores = get_scores(i, pop_weight)

    return np.argsort(-scores).tolist().index(j)

In [232]:
pop_weight = .007
# pop_weight = 0

metrics = [
    get_metric(a, b, pop_weight),
    get_metric(b, a, pop_weight),
    
    get_metric(b, c, pop_weight),
    get_metric(c, b, pop_weight),
]

np.mean(metrics)

C:\Users\johns\AppData\Local\Temp\ipykernel_28352\1226608394.py:3: RuntimeWarning: divide by zero encountered in log2
  scores = scores**pop_weight * np.log2(sppmi[:, i].toarray())
C:\Users\johns\AppData\Local\Temp\ipykernel_28352\1226608394.py:3: RuntimeWarning: invalid value encountered in multiply
  scores = scores**pop_weight * np.log2(sppmi[:, i].toarray())


np.float64(6.75)

In [233]:
top_k = 20

similarity_scores = get_scores(a, pop_weight)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

C:\Users\johns\AppData\Local\Temp\ipykernel_28352\1226608394.py:3: RuntimeWarning: divide by zero encountered in log2
  scores = scores**pop_weight * np.log2(sppmi[:, i].toarray())
C:\Users\johns\AppData\Local\Temp\ipykernel_28352\1226608394.py:3: RuntimeWarning: invalid value encountered in multiply
  scores = scores**pop_weight * np.log2(sppmi[:, i].toarray())


['Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)',
 'Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)',
 'Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)',
 'Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)',
 'Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)',
 'S. Carey (spotify:artist:2LSJrlndCuTpdEluvYHc2E)',
 'Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'The Acid (spotify:artist:0bRtSoJSpQdnbB3dWrWprR)',
 'The Careful Ones (spotify:artist:1DdAoWvETBUklcJCOISZx1)',
 'SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)',
 'Snakadaktal (spotify:artist:0SdEkx5Ai2gl0W7pnhlsfy)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)',
 'Matthew And The Atlas (spotify:artist:0lSENl3bteP8p2NbiSP7RM)',
 'Matt Corby (spotify:artist:7CIW23FQUXPc1zebnO1TDG)',
 'Vancouver Sleep Clinic (spotify:artist:77BznF1Dr1k5KyEZ

In [143]:
top_k = 20

similarity_scores = get_scores(b, pop_weight)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

C:\Users\johns\AppData\Local\Temp\ipykernel_28352\2848318376.py:2: RuntimeWarning: divide by zero encountered in log2
  scores = np.log2(PXY[:, i].toarray())


['Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Alice Boman (spotify:artist:3WiytRnvoL0kT3oAGl9TCt)',
 'Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Amason (spotify:artist:4cJKxS7uOPhwb5UQ70sYpN)',
 'SOAK (spotify:artist:4PLsMEk2DCRVlVL2a9aZAv)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Oh Pep! (spotify:artist:3L9rqEIsNSaOcx2QIstn7v)',
 'Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)',
 'Farao (spotif

In [144]:
top_k = 20

similarity_scores = get_scores(c, pop_weight)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

C:\Users\johns\AppData\Local\Temp\ipykernel_28352\2848318376.py:2: RuntimeWarning: divide by zero encountered in log2
  scores = np.log2(PXY[:, i].toarray())


['Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'No. 4 (spotify:artist:24YjyPpqFQi1Oh7PQSBT3J)',
 'Cezinando (spotify:artist:504cl42JQLRqlZddfZ3S4z)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'A-Laget (spotify:artist:2eJz0fbC9ZqBL3mGWPBvI5)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'TIX (spotify:artist:6CawoDDP1IZUSGl4wSJGC9)',
 'Amanda Delara (spotify:artist:2hxV1sx7MESHOuxBi04hU6)',
 'Conan Gray (spotify:artist:4Uc8Dsxct0oMqx0P6i60ea)',
 'Bodø Domkor (spotify:artist:6QQCeD7ZErV1xQnlyjaFHF)',
 'Surferosa (spotify:artist:5WUYimkWakv41ZVVIq3BDr)',
 'Herreløse (spotify:artist:3xvgxCYu3z6OR0MN1g1Nlu)',
 'Bendik (spotify:artist:4krYRNHjKcETSEY2Ghf9Mo)',
 'Pieces of Juno (spot